In [ ]:
import json
def get_tr_vl_ts_list(dataset_dtl,fold=0):

    with open(dataset_dtl, 'r') as f:
        fold_data = json.load(f)
    test_list = []
    for name in fold_data['test_files']:
        test_list.append(name.split('.nii.gz')[0])
    # test_list = [n.lower() for n in test_list]

    train_list =[]
    for name in fold_data['cross_validation'][fold]['train_files']:
        train_list.append(name.split('.nii.gz')[0])
    # train_list = [n.lower() for n in train_list]


    val_list=[]
    for name in fold_data['cross_validation'][fold]['validation_files']:
        val_list.append(name.split('.nii.gz')[0])
    # val_list = [n.lower() for n in val_list]
    
    return train_list,val_list,test_list

In [ ]:
from joblib import load
import os
import lime
import lime.lime_tabular
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

In [ ]:
def get_x_test(dataset_dtl_path,fold,data,feature_list):
    train_list,val_list,test_list = get_tr_vl_ts_list(dataset_dtl=dataset_dtl_path,fold=fold)
    train_val_data = data[data['Name'].isin(val_list+train_list)]

    # Use train and val data to build scalar
    standard_scaler = StandardScaler()
    train_val_data.drop(columns=['Center','Name','Label'])
    x_train_val_raw = train_val_data.drop(columns=['Center','Name','Label'])
    y_train_val = train_val_data['Label']
    standard_scaler.fit_transform(x_train_val_raw)

    test_data = data[data['Name'].isin(test_list)]
    x_test_raw = test_data.drop(columns=['Center','Name','Label'])
    y_test = test_data[["Label"]]
    x_test_scaled = pd.DataFrame(standard_scaler.transform(x_test_raw),columns=x_test_raw.columns)
    x_test = x_test_scaled[feature_list]
    x_test = x_test[feature_list]
    return x_test, x_train_val_raw, standard_scaler

In [ ]:
feature_dim = '2D'
train_test_info = 'Train_Test_4'
if feature_dim=='2D':
    if train_test_info =='Train_Test_4':
        # Test 4
        feature_list = [
        'skewness-Laws R5S5',
        'Collage_kurt_MaximalCorrelationCoefficient_1_nb_4_ws_5',
        'Collage_skew_InformationMeasureOfCorrelation2_1_nb_8_ws_3',
        'Collage_kurt_Contrast_1_nb_8_ws_7',
        'Collage_median_MaximalCorrelationCoefficient_1_nb_32_ws_3',
        'Collage_skew_Correlation_1_nb_16_ws_5',
        'Collage_skew_MaximalCorrelationCoefficient_1_nb_4_ws_3',
        'Collage_skew_SumVariance_1_nb_8_ws_5',
        'median-Laws E5L5',
        'Collage_skew_Entropy_1_nb_16_ws_7',
        'skewness-Laws S5E5',
        'skewness-Laws W5L5',
        'Collage_skew_DifferenceEntropy_1_nb_4_ws_3',
        'Collage_var_Contrast_1_nb_8_ws_7',
        'Collage_kurt_SumEntropy_1_nb_16_ws_5',
        ]
    elif train_test_info =='Train_Test_3':
        # test3
        feature_list = ['Collage_skew_InformationMeasureOfCorrelation2_1_nb_8_ws_3',
        'skewness-Laws S5E5',
        'skewness-Laws R5S5',
        'Collage_skew_InformationMeasureOfCorrelation2_1_nb_16_ws_5',
        'Collage_skew_Correlation_1_nb_8_ws_5',
        'median-Laws E5L5',
        'median-Laws W5S5',
        'Collage_skew_MaximalCorrelationCoefficient_1_nb_4_ws_3']
    elif train_test_info =='Train_Test_2':
        # test2
        feature_list = ['Collage_kurt_MaximalCorrelationCoefficient_1_nb_4_ws_5',
        'median-Laws E5L5',
        'Collage_skew_InformationMeasureOfCorrelation2_1_nb_8_ws_3',
        'Collage_var_InformationMeasureOfCorrelation2_1_nb_16_ws_3',
        'skewness-Laws R5S5',
        'Collage_skew_MaximalCorrelationCoefficient_1_nb_4_ws_3',
        'skewness-Laws S5E5',
        'median-Laws W5S5',
        'Collage_kurt_MaximalCorrelationCoefficient_1_nb_4_ws_7',
        'Collage_median_SumEntropy_1_nb_8_ws_3',
        'Collage_kurt_SumEntropy_1_nb_16_ws_5',
        'Collage_skew_SumVariance_1_nb_4_ws_7']
    elif train_test_info =='Train_Test_1':
        feature_list = ['Collage_skew_InformationMeasureOfCorrelation2_1_nb_8_ws_3',
        'Collage_var_InformationMeasureOfCorrelation2_1_nb_16_ws_3',
        'Collage_kurt_Contrast_1_nb_16_ws_7',
        'Collage_skew_MaximalCorrelationCoefficient_1_nb_4_ws_3',
        'Collage_kurt_SumAverage_1_nb_16_ws_3',
        'Collage_kurt_InformationMeasureOfCorrelation2_1_nb_16_ws_3',
        'Collage_kurt_MaximalCorrelationCoefficient_1_nb_4_ws_5',
        'median-Laws E5L5',
        'Collage_kurt_SumEntropy_1_nb_16_ws_3',
        'skewness-Laws S5E5',
        'kurtosis-Haralick correlation ws=7 n=4',
        'Collage_kurt_InformationMeasureOfCorrelation2_1_nb_64_ws_5',
        'Collage_median_DifferenceVariance_1_nb_64_ws_7',
        'skewness-Laws E5S5',
        'skewness-Gradient sobelxy']
elif feature_dim=='3D':
    if train_test_info =='Train_Test_1':
        feature_list = [
        'Collage_var_InformationMeasureOfCorrelation2_1_nb_8_ws_5',
        'Collage_var_SumEntropy_1_nb_32_ws_7',
        'skewness-Haralick info1 ws=5 n=4',
        'skewness-Laws  S5L5E5',
        'median-Laws  E5E5E5',
        'Collage_skew_DifferenceEntropy_1_nb_32_ws_7',
        'median-Laws  R5E5S5',
        'median-Laws  W5E5S5',
        'Collage_kurt_InformationMeasureOfCorrelation2_1_nb_4_ws_5',
        'median-Laws  E5L5S5',
                ]
    elif train_test_info =='Train_Test_2':
        # test3
        feature_list = ['Collage_var_InformationMeasureOfCorrelation2_1_nb_16_ws_3',
        'median-Laws  E5L5S5',
        'median-Laws  E5E5E5',
        'skewness-Laws  R5R5L5',
        'median-Laws  L3S3L3',
        'skewness-Laws  S3E3S3',
        'Collage_kurt_Correlation_2_nb_64_ws_5',
        'Collage_kurt_SumAverage_1_nb_32_ws_7',
        'skewness-Laws  S5L5E5',
        'skewness-Laws  S3E3E3',
        'skewness-Laws  W5L5S5',
        'kurtosis-Laws  L5R5L5',]
    elif train_test_info =='Train_Test_3':
        # test2
        feature_list = ['skewness-Laws  S5W5R5',
        'skewness-Laws  S5L5E5',
        'median-Laws  L3S3L3',
        'median-Laws  S3E3E3',
        'Collage_skew_MaximalCorrelationCoefficient_1_nb_4_ws_5',
        'median-Laws  R5E5S5',
        'skewness-Laws  W5L5S5',
        'Collage_kurt_SumAverage_1_nb_32_ws_7',
        'Collage_var_InformationMeasureOfCorrelation2_1_nb_16_ws_3',
        'median-Laws  E5L5S5',
        'Collage_kurt_InformationMeasureOfCorrelation2_1_nb_4_ws_5',
        'Collage_kurt_InformationMeasureOfCorrelation2_2_nb_16_ws_7',
        'Collage_kurt_InformationMeasureOfCorrelation2_1_nb_16_ws_3',
        'median-Laws  E5E5E5',
        'median-Laws  W5E5S5',]
    elif train_test_info =='Train_Test_4':
        feature_list = ['skewness-Laws  E5E5R5',
        'skewness-Laws  E3S3E3',
        'skewness-Laws  R5E5E5',
        'skewness-Haralick correlation ws=5 n=4',
        'skewness-Laws  E3E3E3',
        'Collage_kurt_SumAverage_1_nb_16_ws_7',
        'Collage_kurt_MaximalCorrelationCoefficient_1_nb_4_ws_7',
        'skewness-Haralick info1 ws=5 n=64',
        'Collage_kurt_InformationMeasureOfCorrelation2_2_nb_16_ws_7',
        'skewness-Laws  R5R5L5',
        'Collage_var_SumEntropy_1_nb_32_ws_7',
        'median-Laws  R5E5L5']

In [ ]:
# Aggregate LIME importance across all folds
all_lime_importances = defaultdict(list)

for fold in range(5):
    feature_path = '/home/pyq6817/IPMN-Radiomics-Plus-Deeplearning/radiomics'
    dataset_dtl_path = f'/home/pyq6817/IPMN-Radiomics-Plus-Deeplearning/{train_test_info}.json'
    radiomics_model_dir = f'/home/pyq6817/IPMN-Radiomics-Plus-Deeplearning/radiomics/trained_models/{feature_dim}'
    
    # Load model
    radiomics_model = load(os.path.join(radiomics_model_dir,f'{train_test_info}/{train_test_info}_Fold{fold+1}.joblib'))
    
    # Load data
    data = pd.read_csv(os.path.join(feature_path,f'Data/{feature_dim.upper()}_t2/all.csv'))
    x_test, x_train_val_raw, scaler = get_x_test(dataset_dtl_path,fold,data,feature_list)
    
    # Prepare training data for LIME (use train+val data for better local approximation)
    train_list, val_list, test_list = get_tr_vl_ts_list(dataset_dtl=dataset_dtl_path, fold=fold)
    train_val_data = data[data['Name'].isin(val_list+train_list)]
    x_train_val_raw = train_val_data.drop(columns=['Center','Name','Label'])
    x_train_val_scaled = pd.DataFrame(scaler.transform(x_train_val_raw), columns=x_train_val_raw.columns)
    x_train_val = x_train_val_scaled[feature_list].values
    
    # Create LIME explainer
    explainer = lime.lime_tabular.LimeTabularExplainer(
        x_train_val,
        feature_names=feature_list,
        class_names=['Low Risk', 'High Risk'],
        mode='classification',
        discretize_continuous=True
    )
    
    # Calculate LIME importance for each test sample
    fold_importances = defaultdict(list)
    
    for idx in range(len(x_test)):
        instance = x_test.iloc[idx].values
        
        # Get explanation
        explanation = explainer.explain_instance(
            instance,
            radiomics_model.predict_proba,
            num_features=len(feature_list),
            top_labels=1
        )
        
        # Extract feature importance for the predicted class
        exp_list = explanation.as_list(label=1)  # Get explanation for class 1 (High Risk)
        
        # Store importances
        for feature_name, importance in exp_list:
            fold_importances[feature_name].append(abs(importance))
            all_lime_importances[feature_name].append(abs(importance))
    
    print(f"Fold {fold+1} completed. Processed {len(x_test)} test samples.")

In [ ]:
# Calculate average importance for each feature
avg_importance = {}
for feature_name, importances in all_lime_importances.items():
    avg_importance[feature_name] = np.mean(importances)

# Sort features by average importance
sorted_features = sorted(avg_importance.items(), key=lambda x: x[1], reverse=True)

print("Average LIME Feature Importance (across all folds and samples):")
print("-" * 60)
for feature_name, importance in sorted_features:
    print(f"{feature_name:50s}: {importance:.4f}")

In [ ]:
# Visualize feature importance
plt.figure(figsize=(12, 8))
features = [f[0] for f in sorted_features]
importances = [f[1] for f in sorted_features]

# Create horizontal bar plot
y_pos = np.arange(len(features))
plt.barh(y_pos, importances, align='center')
plt.yticks(y_pos, features, fontsize=10)
plt.xlabel('Average LIME Importance', fontsize=12)
plt.title(f'LIME Feature Importance - {feature_dim} {train_test_info}', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()  # Top feature at top
plt.tight_layout()

# Save figure
os.makedirs(f'/home/pyq6817/IPMN-Radiomics-Plus-Deeplearning/radiomics/LIME', exist_ok=True)
plt.savefig(f'/home/pyq6817/IPMN-Radiomics-Plus-Deeplearning/radiomics/LIME/{feature_dim}_{train_test_info}_avg.png', 
            bbox_inches='tight', dpi=300)
plt.show()
plt.clf()

In [ ]:
# Analyze each fold separately
for fold in range(5):
    feature_path = '/home/pyq6817/IPMN-Radiomics-Plus-Deeplearning/radiomics'
    dataset_dtl_path = f'/home/pyq6817/IPMN-Radiomics-Plus-Deeplearning/{train_test_info}.json'
    radiomics_model_dir = f'/home/pyq6817/IPMN-Radiomics-Plus-Deeplearning/radiomics/trained_models/{feature_dim}'
    
    # Load model
    radiomics_model = load(os.path.join(radiomics_model_dir,f'{train_test_info}/{train_test_info}_Fold{fold+1}.joblib'))
    
    # Load data
    data = pd.read_csv(os.path.join(feature_path,f'Data/{feature_dim.upper()}_t2/all.csv'))
    x_test, x_train_val_raw, scaler = get_x_test(dataset_dtl_path,fold,data,feature_list)
    
    # Prepare training data for LIME
    train_list, val_list, test_list = get_tr_vl_ts_list(dataset_dtl=dataset_dtl_path, fold=fold)
    train_val_data = data[data['Name'].isin(val_list+train_list)]
    x_train_val_raw = train_val_data.drop(columns=['Center','Name','Label'])
    x_train_val_scaled = pd.DataFrame(scaler.transform(x_train_val_raw), columns=x_train_val_raw.columns)
    x_train_val = x_train_val_scaled[feature_list].values
    
    # Create LIME explainer
    explainer = lime.lime_tabular.LimeTabularExplainer(
        x_train_val,
        feature_names=feature_list,
        class_names=['Low Risk', 'High Risk'],
        mode='classification',
        discretize_continuous=True
    )
    
    # Calculate LIME importance for each test sample
    fold_importances = defaultdict(list)
    
    for idx in range(len(x_test)):
        instance = x_test.iloc[idx].values
        
        # Get explanation
        explanation = explainer.explain_instance(
            instance,
            radiomics_model.predict_proba,
            num_features=len(feature_list),
            top_labels=1
        )
        
        # Extract feature importance for the predicted class
        exp_list = explanation.as_list(label=1)
        
        # Store importances
        for feature_name, importance in exp_list:
            fold_importances[feature_name].append(abs(importance))
    
    # Calculate average importance for this fold
    fold_avg_importance = {}
    for feature_name, importances in fold_importances.items():
        fold_avg_importance[feature_name] = np.mean(importances)
    
    # Sort features by average importance
    sorted_features_fold = sorted(fold_avg_importance.items(), key=lambda x: x[1], reverse=True)
    
    # Visualize
    plt.figure(figsize=(12, 8))
    features = [f[0] for f in sorted_features_fold]
    importances = [f[1] for f in sorted_features_fold]
    
    y_pos = np.arange(len(features))
    plt.barh(y_pos, importances, align='center')
    plt.yticks(y_pos, features, fontsize=10)
    plt.xlabel('Average LIME Importance', fontsize=12)
    plt.title(f'LIME Feature Importance - {feature_dim} {train_test_info} Fold {fold+1}', 
              fontsize=14, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    
    # Save figure
    os.makedirs(f'/home/pyq6817/IPMN-Radiomics-Plus-Deeplearning/radiomics/LIME/{feature_dim}', exist_ok=True)
    plt.savefig(f'/home/pyq6817/IPMN-Radiomics-Plus-Deeplearning/radiomics/LIME/{feature_dim}/{train_test_info}_fold{fold}.png', 
                bbox_inches='tight', dpi=300)
    plt.show()
    plt.clf()
    
    print(f"Fold {fold+1} analysis completed and saved.")

In [ ]:
# Example: Analyze a specific sample from fold 0
fold = 0
sample_idx = 0  # Change this to analyze different samples

feature_path = '/home/pyq6817/IPMN-Radiomics-Plus-Deeplearning/radiomics'
dataset_dtl_path = f'/home/pyq6817/IPMN-Radiomics-Plus-Deeplearning/{train_test_info}.json'
radiomics_model_dir = f'/home/pyq6817/IPMN-Radiomics-Plus-Deeplearning/radiomics/trained_models/{feature_dim}'

# Load model
radiomics_model = load(os.path.join(radiomics_model_dir,f'{train_test_info}/{train_test_info}_Fold{fold+1}.joblib'))

# Load data
data = pd.read_csv(os.path.join(feature_path,f'Data/{feature_dim.upper()}_t2/all.csv'))
x_test, x_train_val_raw, scaler = get_x_test(dataset_dtl_path,fold,data,feature_list)

# Prepare training data for LIME
train_list, val_list, test_list = get_tr_vl_ts_list(dataset_dtl=dataset_dtl_path, fold=fold)
train_val_data = data[data['Name'].isin(val_list+train_list)]
x_train_val_raw = train_val_data.drop(columns=['Center','Name','Label'])
x_train_val_scaled = pd.DataFrame(scaler.transform(x_train_val_raw), columns=x_train_val_raw.columns)
x_train_val = x_train_val_scaled[feature_list].values

# Create LIME explainer
explainer = lime.lime_tabular.LimeTabularExplainer(
    x_train_val,
    feature_names=feature_list,
    class_names=['Low Risk', 'High Risk'],
    mode='classification',
    discretize_continuous=True
)

# Get explanation for specific sample
instance = x_test.iloc[sample_idx].values
prediction = radiomics_model.predict_proba(instance.reshape(1, -1))[0]

print(f"Sample {sample_idx} prediction probabilities:")
print(f"  Low Risk:  {prediction[0]:.4f}")
print(f"  High Risk: {prediction[1]:.4f}")
print(f"  Predicted class: {'High Risk' if prediction[1] > prediction[0] else 'Low Risk'}")
print("\n" + "="*60)

# Get explanation
explanation = explainer.explain_instance(
    instance,
    radiomics_model.predict_proba,
    num_features=len(feature_list),
    top_labels=1
)

# Show explanation
explanation.show_in_notebook(show_table=True, show_all=False)

In [ ]:
# Save feature importance to CSV
importance_df = pd.DataFrame(sorted_features, columns=['Feature', 'Average_Importance'])
importance_df = importance_df.sort_values('Average_Importance', ascending=False)

os.makedirs(f'/home/pyq6817/IPMN-Radiomics-Plus-Deeplearning/radiomics/LIME', exist_ok=True)
importance_df.to_csv(f'/home/pyq6817/IPMN-Radiomics-Plus-Deeplearning/radiomics/LIME/{feature_dim}_{train_test_info}_importance.csv', 
                     index=False)

print(f"Feature importance saved to CSV.")
print(importance_df.head(10))